In [3]:
!pip install -U transformers -q

In [4]:
# Load model directly
from transformers import AutoProcessor, AutoModelForImageTextToText, AutoTokenizer

LLM_ID = "Qwen/Qwen3.5-4B"

processor = AutoProcessor.from_pretrained(LLM_ID)
model = AutoModelForImageTextToText.from_pretrained(LLM_ID, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(LLM_ID)
model.generation_config.pad_token_id = tokenizer.pad_token_id

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [5]:
def create_message(content, role="user"):
    '''
    This function lets you format text input into the format a LLM requires.
    
    role - The entity the message corresponds to. Should be one of 'user', 'helper', or 'system'.
    content - The actual content of the message being sent. Must be text.
    '''
    msg = {
        "role": role, 
        "content": [
            {"type": "text", "text": content},
        ],
    }

    return msg

In [6]:
def generate_response(msgs, max_new_tokens=1000):
    '''
    This function lets you input llm formatted messages and receive human readable llm output as 
    the result. Hyperparameter defaults are based on the recommendations for a conversational 
    qwen-3.5-4B model. 

    msgs - list containing the message(s) llm will use as the basis for generating its output. 
    max_new_tokens - The max amount of generated tokens prior to the LLM cutting off its output.
    
    '''
    llm_input = processor.apply_chat_template(
    	msgs,
    	add_generation_prompt=True,
    	tokenize=True,
    	return_dict=True,
    	return_tensors="pt",
        enable_thinking=False,
    ).to(model.device)


    llm_output = model.generate(
        **llm_input, 
        max_new_tokens=max_new_tokens, 
        do_sample=True,
        temperature=.7, 
        top_p=0.8, 
        top_k=20, 
        min_p=0.0, 
        repetition_penalty=1.0,
    )

    readable_output = processor.decode(llm_output[0][llm_input["input_ids"].shape[-1]:], skip_special_tokens=True)

    return readable_output

In [7]:
def predict_trait(sys_prompt_msg, text):
    
    text = create_message(text)
    conversation = [sys_prompt_msg] + [text]   
    prediction = generate_response(conversation).strip()
    if not prediction.isdigit():
        prediction = -1 ## Error
    prediction = int(prediction)

    return prediction

In [14]:
import pandas as pd
import torch
def load_data(input_csv, prior_predictions_csv, output_csv):
   
    df_input = pd.read_csv(input_csv, dtype=str).fillna("")
    
    # Get llm output columns (e.g. llm_output_0 .. llm_output_4)
    output_cols = sorted([c for c in df_input.columns if c.startswith('llm_output_')],
                         key=lambda s: int(s.split('_')[-1]) if s.split('_')[-1].isdigit() else s)

    items = []  # list of (data_id, num_turns, label, concat_text)
    prior_data_ids = []

    if os.path.exists(output_csv):
        prior_df = pd.read_csv(output_csv, dtype=str).fillna("")
        prior_data_ids = prior_df['data_id'].tolist()

    elif os.path.exists(prior_predictions_csv):
        prior_df = pd.read_csv(prior_predictions_csv, dtype=str).fillna("")
        prior_data_ids = prior_df['data_id'].tolist()
        prior_df.to_csv(output_csv, mode='w', index=False)
        
    for _, row in df_input.iterrows():
        conversation = []
        data_id = row.get('data_id')
        
        if data_id not in prior_data_ids:
            # Try trait_selected, fallback to trait or '-1' for error
            trait = row.get('trait_selected', row.get('trait', '-1'))
            try:
                label = int(trait) if trait != '' else -1
            except Exception:
                label = -1
        
            outputs = []
            for col in output_cols:
                txt = row.get(col, "")
                if txt is None:
                    txt = ""
                txt = str(txt).strip()
                if txt != "":
                    outputs.append(txt)
        
            if len(outputs) == 0:
                continue
        
            # create concatenated texts for each convo length, include num_turns in each item
            for k in range(1, len(outputs) + 1):
                concat = "\n\n".join(outputs[:k])
                conversation.append((data_id, k, label, concat))  # (data_id, num_turns, label, text)
            
            items.append(conversation)

    return items

In [15]:
import os

def add_to_dataset(predicted_conversation, output_csv):
    formatted_list = []
    
    for prediction_data in predicted_conversation:
        data = {
            'data_id': prediction_data[0],
            'conversation_turn': prediction_data[1],
            'label': prediction_data[2], 
            'prediction': prediction_data[3],
            'system_prompt': [prediction_data[4]],            
            'text': prediction_data[5],            
            }
        formatted_list.append(data)

    df = pd.DataFrame(formatted_list)
    df.to_csv(output_csv, mode='a', header=not os.path.exists(output_csv), index=False)

In [10]:
from tqdm import tqdm

def load_and_predict(input_csv, prior_predictions_csv, sys_prompt, output_csv):    
    items = load_data(input_csv, prior_predictions_csv, output_csv)
    sys_prompt_msg = create_message(sys_prompt, role="system")

    for item in tqdm(items):
        predicted_conversation = []
        
        for conversation in item:
            data_id, k, lab, text = conversation
            
            # Make the predictions
            prediction = predict_trait(sys_prompt_msg, text)
            
            prediction_data = data_id, k, lab, prediction, sys_prompt, text
            predicted_conversation.append(prediction_data)
        
        add_to_dataset(predicted_conversation, output_csv)

## Zero-Shot prompting

In [11]:
sys_prompt = """You will be provided with the generated output from a LLM conversation.

It is your purpose to determine which trait the user asking the question in each of those 
conversations had.

There might be one instance of output generated in response to the user or there may be several. 
Regardless, all output corresponds to the same user and the responses should be treated as data
points towards predicting which trait the user has.

The potential traits users can have are: 

0. None
1. The user is very budget-conscious and prefers low-cost or highly affordable options.
2. The user has moderate spending flexibility and values a balance between cost and quality.
3. The user prioritizes premium experiences and is not sensitive to price.

Respond with the number (0, 1, 2, or 3) corresponding to the trait the user is most likely to
have and nothing else. 
"""

input_csv = "/kaggle/input/datasets/zacharyknox/synthetic-conversations-full/synthetic_conversations_full.csv"
prior_predictions_csv = "/kaggle/input/datasets/zacharyknox/zero-shot-llm-observer/zero_shot_llm_observer.csv"
output_csv = "zero_shot_llm_observer.csv"

In [12]:
# Test data run:
load_and_predict(input_csv, prior_predictions_csv, sys_prompt, output_csv)


0it [00:00, ?it/s]


## Few Shot Prompting

In [13]:
sys_prompt = """You will be provided with the generated output from a LLM conversation.

It is your purpose to determine which trait the user asking the question in each of those 
conversations had.

There might be one instance of output generated in response to the user or there may be several. 
Regardless, all output corresponds to the same user and the responses should be treated as data
points towards predicting which trait the user has.

The potential traits users can have are: 

0. None
1. The user is very budget-conscious and prefers low-cost or highly affordable options.
2. The user has moderate spending flexibility and values a balance between cost and quality.
3. The user prioritizes premium experiences and is not sensitive to price.

--------

An example of a single generated LLM response which corresponds to a user with no designated trait
-- option 0:

For video editing under $1,200, the Lenovo IdeaPad Slim 5 Pro with an Intel Core i7 processor and 16GB RAM offers strong performance while remaining portable at 14 inches. The trade-off is that it lacks dedicated graphics hardware, which may limit rendering speed for 4K footage compared to models with RTX cards, though it remains sufficient for 1080p workloads.

Alternatively, the Dell XPS 13 Plus provides a premium chassis and better portability with a 13-inch display, but its base configuration often requires upgrading to 16GB RAM and a higher-tier processor to meet your budget, potentially reducing the funds available for storage. If you prioritize screen quality and build over raw processing power, the XPS is viable; if you prioritize editing speed within the budget, the Lenovo offers a more balanced configuration without needing immediate component upgrades.

--------

An example of a single generated LLM response which corresponds to a user with the designated trait of 
"The user is very budget-conscious and prefers low-cost or highly affordable options" -- option 1:

For a budget of $1,200, the Samsung Galaxy S24 Ultra is the more viable option, as the iPhone 15 Pro typically exceeds this price point even with discounts. The S24 Ultra offers a 200MP main sensor and 100x Space Zoom, providing superior versatility for distant subjects in travel scenarios, whereas the iPhone 15 Pro's 5x optical zoom may require cropping for distant shots.

However, the iPhone 15 Pro provides a more consistent color science and better low-light performance in dynamic scenes, which can be advantageous for street photography. If the primary constraint is strictly adhering to the $1,200 limit, the S24 Ultra is the recommended choice, but you should verify current pricing as the iPhone 15 Pro may occasionally drop close to this threshold during sales events.

--------

An example of a single generated LLM response which corresponds to a user with the designated trait of 
"The user has moderate spending flexibility and values a balance between cost and quality." -- option
2:

For a laptop under $800 suited for office tasks and video calls, the Dell Latitude 3420 offers robust build quality and reliable performance at approximately $650. In contrast, the Acer Aspire 5 provides a similar price point around $600 but features a less durable chassis and slightly lower screen brightness.

The primary trade-off here is longevity and display clarity: the Dell's metal construction and 100% sRGB color coverage are better for professional consistency, while the Acer's lower cost allows for a larger 15.6-inch display if screen real estate is prioritized over portability. Given your requirement for a balance between cost and quality, the Dell is the stronger choice for sustained daily use, though the Acer remains a viable option if budget is the absolute limiting factor.

--------

An example of a single generated LLM response which corresponds to a user with the designated trait of 
"The user prioritizes premium experiences and is not sensitive to price" -- option 3:

Given your priority for premium experiences, the MacBook Pro 16-inch offers exceptional battery life (up to 22 hours) and high-end performance, though it commands a higher price point. Alternatively, the Dell XPS 15 provides similar portability and display quality with slightly lower battery endurance but often at a more competitive cost.

Considering your lack of price sensitivity, the MacBook Pro's superior thermal management and longer battery duration may offer a more seamless workflow for extended remote sessions, whereas the Dell XPS might serve as a cost-effective entry into the premium tier if budget flexibility allows for a trade-off in battery longevity.

--------

You will now receive the actual conversation. Please respond with the number (0, 1, 2, or 3) 
that is most likely to correspond to the trait represented by the generated output and nothing else. 

"""

input_csv = "/kaggle/input/datasets/zacharyknox/synthetic-conversations-full/synthetic_conversations_full.csv"
prior_predictions_csv = "/kaggle/input/datasets/zacharyknox/few-shot-llm-observer/few_shot_llm_observer.csv"
output_csv = "few_shot_llm_observer.csv"

In [ ]:
# Test data run:
load_and_predict(input_csv, prior_predictions_csv, sys_prompt, output_csv)


  4%|▍         | 3/67 [01:33<33:29, 31.40s/it]